Description:

Project statement:

AAL, established in 2000, is a well-known brand in Australia, particularly recognized for its clothing business. It has opened branches in various states, metropolises, and tier-1 and tier-2 cities across the country.

The brand caters to all age groups, from kids to the elderly.

Currently experiencing a surge in business, AAL is actively pursuing expansion opportunities. To facilitate informed investment decisions, the CEO has assigned the responsibility to the head of AAL’s sales and marketing (S&M) department. The specific tasks include:

Identify the states that are generating the highest revenues.
Develop sales programs for states with lower revenues. The head of sales and marketing has requested your assistance with this task.
Analyze the sales data of the company for the fourth quarter in Australia, examining it on a state-by-state basis. Provide insights to assist the company in making data-driven decisions for the upcoming year.

*Enclosed is the CSV (AusApparalSales4thQrt2020.csv) file that covers the said data.

# Libraries

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder


# Step 1: Loading Data

In [25]:
# Load the dataset
aas4qData = pd.read_csv('../../data-sets/course-end-dataset/AusApparalSales4thQrt2020.csv')

In [26]:
# Look at the data
print(f"Rows X Coloumns: {aas4qData.shape}")

Rows X Coloumns: (7560, 6)


In [27]:
# Looking at the data:First 5 rows
aas4qData.head()

,Date,Time,State,Group,Unit,Sales
0,1-Oct-2020,Morning,WA,Kids,8,20000
1,1-Oct-2020,Morning,WA,Men,8,20000
2,1-Oct-2020,Morning,WA,Women,4,10000
3,1-Oct-2020,Morning,WA,Seniors,15,37500
4,1-Oct-2020,Afternoon,WA,Kids,3,7500


In [28]:
# Check for unique values in 'Time', 'State', and 'Group' columns
for col in ['Time', 'State', 'Group']:
    print(f"Unique values in {col} column: {aas4qData[col].unique()}")

Unique values in Time column: <StringArray>
[' Morning', ' Afternoon', ' Evening']
Length: 3, dtype: str
Unique values in State column: <StringArray>
[' WA', ' NT', ' SA', ' VIC', ' QLD', ' NSW', ' TAS']
Length: 7, dtype: str
Unique values in Group column: <StringArray>
[' Kids', ' Men', ' Women', ' Seniors']
Length: 4, dtype: str


In [29]:
# Check for total records for each column
for col in ['Date', 'Time', 'State', 'Group']:
    print(f"Unique values in {col} column: {aas4qData[col].value_counts()}")

Unique values in Date column: Date
1-Oct-2020     84
2-Oct-2020     84
3-Oct-2020     84
4-Oct-2020     84
5-Oct-2020     84
               ..
26-Dec-2020    84
27-Dec-2020    84
28-Dec-2020    84
29-Dec-2020    84
30-Dec-2020    84
Name: count, Length: 90, dtype: int64
Unique values in Time column: Time
Morning      2520
Afternoon    2520
Evening      2520
Name: count, dtype: int64
Unique values in State column: State
WA     1080
NT     1080
SA     1080
VIC    1080
QLD    1080
NSW    1080
TAS    1080
Name: count, dtype: int64
Unique values in Group column: Group
Kids       1890
Men        1890
Women      1890
Seniors    1890
Name: count, dtype: int64


# Step 2: Cleaning Data

In [30]:
# Checking Null Values inside the data
aas4qData.isnull().sum()


Date     0
Time     0
State    0
Group    0
Unit     0
Sales    0
dtype: int64

In [31]:
# Chekcing na value values in the data
aas4qData.isna().sum()

Date     0
Time     0
State    0
Group    0
Unit     0
Sales    0
dtype: int64

In [32]:
# checkign data types of each coloumn
print(aas4qData.dtypes)

Date       str
Time       str
State      str
Group      str
Unit     int64
Sales    int64
dtype: object


In [35]:
#stripping spaces of each coloumn name and also from every text Coloumn: removing spaces would help grouping correctly.
aas4qData.columns = aas4qData.columns.str.strip()

for col in aas4qData.select_dtypes("str"):
    aas4qData[col] = aas4qData[col].str.strip()


In [37]:
# converting Date coloumn to a real data type so we can extract the month later
aas4qData["Date"] = pd.to_datetime(aas4qData['Date'], format="%d-%b-%Y")
aas4qData['Month'] = aas4qData["Date"].dt.month_name()

In [38]:
aas4qData.head()

,Date,Time,State,Group,Unit,Sales,Month
0,2020-10-01,Morning,WA,Kids,8,20000,October
1,2020-10-01,Morning,WA,Men,8,20000,October
2,2020-10-01,Morning,WA,Women,4,10000,October
3,2020-10-01,Morning,WA,Seniors,15,37500,October
4,2020-10-01,Afternoon,WA,Kids,3,7500,October


# Step 3: State Wise Revenue

In [41]:
state_sales = (
    aas4qData.groupby("State")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

print(state_sales.to_string())
print(f"\n✅ Highest revenue state : {state_sales.idxmax()}  (${state_sales.max():,.0f})")
print(f"⚠️  Lowest  revenue state : {state_sales.idxmin()}  (${state_sales.min():,.0f})")

State
VIC    105565000
NSW     74970000
SA      58857500
QLD     33417500
TAS     22760000
NT      22580000
WA      22152500

✅ Highest revenue state : VIC  ($105,565,000)
⚠️  Lowest  revenue state : WA  ($22,152,500)


## Step 4: Deeper Analysis
- sales by Customer group (Kids / Men / Women / Seniors)
- Sales by Time of day (Morning / Afternoon / Evening)
- Pivot table state versus Group
- Monthly Trend (Q4: Oct, Nov, Dec)

In [46]:
# 4a.sales by Customer group (Kids / Men / Women / Seniors)
group_sales = (
    aas4qData.groupby("Group")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("\nsales by Customer Group:")
print(group_sales.to_string())



sales by Customer Group:
Group
Men        85750000
Women      85442500
Kids       85072500
Seniors    84037500


In [51]:
#4b.Sales by Time of day (Morning / Afternoon / Evening)
time_sales = (
    aas4qData.groupby("Time")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("\nSales by Time of the Day:")
print(time_sales.to_string())


Sales by Time of the Day:
Time
Morning      114207500
Afternoon    114007500
Evening      112087500


In [ ]:
#4c.Pivot table state versus Group
#unstack() tunrns the inner group level into Coloumns -> gives 2D table
state_group = aas4qData.groupby(["State", "Group"])["Sales"].sum().unstack()
print("\nSales by State & Customer Group:")
print(state_group.to_string())


Sales by State & Customer Group:
Group      Kids       Men   Seniors     Women
State                                        
NSW    18587500  19022500  18187500  19172500
NT      5700000   5762500   5465000   5652500
QLD     8510000   8392500   8190000   8325000
SA     14515000  14655000  14717500  14970000
TAS     5775000   5757500   5650000   5577500
VIC    26360000  26407500  26315000  26482500
WA      5625000   5752500   5512500   5262500


In [ ]:
#4d. Monthly Trend (Q4: Oct, Nov, Dec)
#reindex() apply the order, the order that stored in the list below
month_order = ["October", "November", "December"]
month_sales = aas4qData.groupby("Month")["Sales"].sum().reindex(month_order)
print("\nMonthly Sales:")
print(month_sales.to_string())


Monthly Sales:
Month
October     114290000
November     90682500
December    135330000
